In [1]:
# Environment setup
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config").is_dir() and (p / "src").is_dir():
            return p
    raise RuntimeError(f"Could not find project root from {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

Project root: /Users/mac/Studying/OTTO/scenario_generator
Working directory: /Users/mac/Studying/OTTO/scenario_generator/notebooks


In [2]:
# Run the pipeline on the seed scenario
from src.pipeline import Pipeline

pipeline = Pipeline(config_dir=PROJECT_ROOT / "config")
result = pipeline.run(
    scenario_path=PROJECT_ROOT / "config" / "scenarios" / "highway_brake_wear_gradual.yaml",
    output_dir=PROJECT_ROOT / "data" / "fault_labels_notebook",
)

print(f"Scenario:      {result.scenario.name}")
print(f"Vehicle:       {result.scenario.vehicle_id}")
print(f"Duration:      {result.scenario.duration_s} s")
print(f"Frames:        {result.n_frames}")
print(f"Signal rows:   {result.n_signal_rows}")
print(f"Applied faults: {len(result.applied_faults)}")
print()
print(f"Output files:")
print(f"  frames:  {result.frames_path.name}")
print(f"  signals: {result.signals_path.name}")
print(f"  faults:  {result.faults_path.name}")
print(f"  labels:  {result.labels_path.name}")

Scenario:      Highway Cruise with Brake Wear
Vehicle:       suv_b
Duration:      90.0 s
Frames:        15300
Signal rows:   9000
Applied faults: 2

Output files:
  frames:  highway_cruise_with_brake_wear.frames.csv
  signals: highway_cruise_with_brake_wear.signals.csv
  faults:  highway_cruise_with_brake_wear.faults.csv
  labels:  highway_cruise_with_brake_wear.labels.csv


In [3]:
# Load all four output files
import pandas as pd

frames_df = pd.read_csv(result.frames_path)
signals_df = pd.read_csv(result.signals_path)
faults_df = pd.read_csv(result.faults_path)
labels_df = pd.read_csv(result.labels_path)

print(f"frames_df:  {frames_df.shape}")
print(f"signals_df: {signals_df.shape}")
print(f"faults_df:  {faults_df.shape}")
print(f"labels_df:  {labels_df.shape}")
print()
print("frames columns:  ", list(frames_df.columns))
print("signals columns: ", list(signals_df.columns))
print("faults columns:  ", list(faults_df.columns))
print("labels columns:  ", list(labels_df.columns))

frames_df:  (15300, 3)
signals_df: (9000, 10)
faults_df:  (2, 10)
labels_df:  (15300, 9)

frames columns:   ['timestamp_ms', 'can_id', 'payload_hex']
signals columns:  ['time_s', 'timestamp_ms', 'engine_rpm', 'throttle_position', 'coolant_temperature', 'brake_pressure', 'brake_pedal', 'wheel_speed', 'vehicle_speed', 'longitudinal_acceleration']
faults columns:   ['fault_id', 'fault_type', 'onset_s', 'end_s', 'severity_start', 'severity_end', 'progression', 'physical_category', 'observable_effect', 'affected_signals']
labels columns:   ['timestamp_ms', 'can_id', 'is_fault', 'fault_count', 'severity_max', 'fault_ids', 'fault_types', 'physical_categories', 'observable_effects']


In [4]:
# Faults.csv contents (per-event table)
print("Per-event fault table:")
print(faults_df.to_string(index=False))

Per-event fault table:
fault_id     fault_type  onset_s  end_s  severity_start  severity_end progression physical_category observable_effect          affected_signals
  wear_1 brake_pad_wear     38.0   82.0             0.0           0.4      linear        mechanical             drift longitudinal_acceleration
tamper_1 brake_pad_wear     85.0   90.0             0.0           0.6        step        mechanical             drift longitudinal_acceleration


In [5]:
# Verify alignment: labels.csv rows match frames.csv rows
assert len(labels_df) == len(frames_df), (
    f"Row count mismatch: {len(labels_df)} vs {len(frames_df)}"
)

# Same timestamps, same CAN IDs, same order
assert (labels_df["timestamp_ms"].values == frames_df["timestamp_ms"].values).all()
assert (labels_df["can_id"].values == frames_df["can_id"].values).all()

print(f"✅ Alignment verified: {len(labels_df)} rows aligned with frames.csv")
print()
print("First 10 rows of labels.csv:")
print(labels_df.head(10).to_string(index=False))

✅ Alignment verified: 15300 rows aligned with frames.csv

First 10 rows of labels.csv:
 timestamp_ms can_id  is_fault  fault_count  severity_max fault_ids fault_types physical_categories observable_effects
            0  0x200         0            0           0.0       NaN         NaN                 NaN                NaN
            0  0x220         0            0           0.0       NaN         NaN                 NaN                NaN
            0  0x2B0         0            0           0.0       NaN         NaN                 NaN                NaN
           10  0x200         0            0           0.0       NaN         NaN                 NaN                NaN
           20  0x200         0            0           0.0       NaN         NaN                 NaN                NaN
           20  0x2B0         0            0           0.0       NaN         NaN                 NaN                NaN
           30  0x200         0            0           0.0       NaN         NaN 

In [6]:
# Summary statistics
n_frames = len(labels_df)
n_faulty = (labels_df["is_fault"] == 1).sum()
n_clean = (labels_df["is_fault"] == 0).sum()

print(f"Total frames:     {n_frames:,}")
print(f"Faulty frames:    {n_faulty:,}  ({100 * n_faulty / n_frames:.2f}%)")
print(f"Clean frames:     {n_clean:,}  ({100 * n_clean / n_frames:.2f}%)")
print()

# Breakdown per fault
print("Frames flagged per fault_id:")
for fault_id in ["wear_1", "tamper_1"]:
    count = labels_df["fault_ids"].str.contains(fault_id, na=False).sum()
    print(f"  {fault_id:10s}  {count:,} frames")

# Breakdown by overlapping state
overlap = (labels_df["fault_count"] >= 2).sum()
print()
print(f"Frames with ≥2 overlapping faults: {overlap:,}")

# Breakdown by CAN ID
print()
print("Faulty frames by CAN ID:")
for cid in sorted(labels_df["can_id"].unique()):
    sub = labels_df[labels_df["can_id"] == cid]
    n_f = (sub["is_fault"] == 1).sum()
    print(f"  {cid}  {n_f:,} / {len(sub):,}  ({100 * n_f / len(sub):.2f}%)")

Total frames:     15,300
Faulty frames:    8,330  (54.44%)
Clean frames:     6,970  (45.56%)

Frames flagged per fault_id:
  wear_1      7,480 frames
  tamper_1    850 frames

Frames with ≥2 overlapping faults: 0

Faulty frames by CAN ID:
  0x200  4,900 / 9,000  (54.44%)
  0x220  980 / 1,800  (54.44%)
  0x2B0  2,450 / 4,500  (54.44%)
